# ROGII Wellbore Geology - Dense (MLP) Training

Train a Multi-Layer Perceptron (Dense) model to predict **TVT** from horizontal well logs.

**Features (12):** `MD, X, Y, Z, ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA, GR, TVT_input`

**Label:** `TVT`

**Runtime**: Kaggle GPU (T4/P100) - Keras 3 + JAX backend

**Author**: Samir Attrah

### 1. Environment & Imports
Setup JAX backend for Keras and import necessary libraries.

In [1]:
# Cell 1: Environment & Imports
import os
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import jax
# Enable JAX float64 precision natively
jax.config.update("jax_enable_x64", True)

import keras
from keras import layers, callbacks, regularizers
import jax.numpy as jnp
import numpy as np
import polars as pl
import glob, pickle, warnings, random
warnings.filterwarnings("ignore")

# Show enough digits to round-trip float64 diagnostics
np.set_printoptions(precision=17, floatmode="unique")

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Keras backend : {keras.backend.backend()}")
print(f"Working dir   : {os.getcwd()}")


2026-06-04 01:03:49.136989: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780524229.175720   60046 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780524229.190188   60046 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Keras version : 3.12.0
Keras backend : jax
Working dir   : /home/samer/Documents/competitions/ROGII/notebooks


### 2. Auto-detect Dataset Path
Locate the competition dataset in local or Kaggle environments.

In [2]:
# Cell 2: Auto-detect dataset path

def find_data_dir():
    """Searches common Kaggle mount points for the ROGII dataset."""
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]

    for scan_root in ["/kaggle/input", "/kaggle/input/competitions"]:
        if os.path.isdir(scan_root):
            for entry in os.listdir(scan_root):
                full = os.path.join(scan_root, entry)
                if os.path.isdir(full) and full not in candidates:
                    candidates.append(full)

    print("Searching for ROGII dataset...")
    for path in candidates:
        if not os.path.isdir(path):
            continue

        contents = os.listdir(path)
        has_train = "train" in contents and os.path.isdir(os.path.join(path, "train"))
        n_train = 0
        if has_train:
            n_train = len(glob.glob(os.path.join(path, "train", "*__horizontal_well.csv")))

        if n_train > 0:
            print(f"  V Using {path} (found {n_train} train wells)")
            return path

    raise FileNotFoundError("Could not find ROGII dataset.")

DATA_DIR = find_data_dir()


Searching for ROGII dataset...
  V Using /home/samer/Documents/competitions/ROGII/dataset (found 773 train wells)


### 3. Configuration
Define hyperparameters, model paths, and feature columns.

In [ ]:
# Cell 3: Configuration

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle") else os.path.abspath(os.path.join(os.getcwd(), "..", "outputs"))
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    "seed": 42,
    "data_dir": DATA_DIR,
    "model_path": f"{OUT_DIR}/dense_tvt_model.keras",
    "scaler_path": f"{OUT_DIR}/dense_scaler_params.pkl",
    "hidden_layers": [128, 64, 32, 16],
    "dropout": 0.20,
    "kr_rate": 1e-5,
    "epochs": 50,
    "batch_size": 128,
    "lr": 1e-3,
    "val_ratio": 0.20,
    "max_wells": None,
    "gcn": 1.0,
}

FEATURE_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
TARGET = "TVT"

print(f"Data dir   : {CONFIG['data_dir']}")
print(f"Model path : {CONFIG['model_path']}")
print(f"Features   : {FEATURE_COLS}")


Data dir   : /home/samer/Documents/competitions/ROGII/dataset
Model path : /home/samer/Documents/competitions/ROGII/outputs/dense_tvt_model.keras
Features   : ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']


### 4. Data Helpers & Preparation
Functions for loading, preprocessing, and preparing the flat dataset for Dense training.

In [4]:
# Cell 4: Data helpers

def load_well(data_dir, wid, split="train"):
    """Loads one horizontal well CSV as Polars DataFrame."""
    path = os.path.join(data_dir, split, f"{wid}__horizontal_well.csv")
    return pl.read_csv(path, infer_schema_length=10000)

def preprocess(df):
    """Preprocess: Interpolation and filling nulls for all features."""
    for col in FEATURE_COLS:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).interpolate()
                  .fill_null(strategy="forward").fill_null(strategy="backward")
                  .fill_null(0.0)
            )
    return df

def prepare_data(cfg):
    """Load all wells, concatenate into one large flat dataset (no sequences for Dense)."""
    pattern = os.path.join(cfg["data_dir"], "train", "*__horizontal_well.csv")
    ids_all = sorted(os.path.basename(f).split("__")[0] for f in glob.glob(pattern))
    
    if cfg["max_wells"]:
        ids_all = ids_all[:cfg["max_wells"]]

    np.random.seed(cfg["seed"])
    n_val = max(1, int(len(ids_all) * cfg["val_ratio"]))
    val_set = set(np.random.permutation(len(ids_all))[:n_val])
    
    print(f"Loading {len(ids_all)} wells...")
    
    def get_arrays(ids):
        feats, tgts = [], []
        for wid in ids:
            try:
                df = preprocess(load_well(cfg["data_dir"], wid))
                df = df.filter(pl.col(TARGET).is_not_null())
                if len(df) == 0: continue
                
                feats.append(df.select(FEATURE_COLS).to_numpy().astype(np.float64))
                tgts.append(df.select(TARGET).to_numpy().ravel().astype(np.float64))
            except Exception as e:
                print(f"  Skip {wid}: {e}")
        
        return np.concatenate(feats), np.concatenate(tgts)

    train_ids = [wid for i, wid in enumerate(ids_all) if i not in val_set]
    val_ids = [wid for i, wid in enumerate(ids_all) if i in val_set]
    
    X_train_raw, y_train_raw = get_arrays(train_ids)
    X_val_raw, y_val_raw = get_arrays(val_ids)
    
    # Calculate normalization stats in float64 using JAX
    feat_mean = jnp.mean(jnp.array(X_train_raw, dtype=jnp.float64), axis=0)
    feat_std = jnp.std(jnp.array(X_train_raw, dtype=jnp.float64), axis=0)
    feat_std = jnp.where(feat_std == 0, 1.0, feat_std)
    
    target_mean = jnp.mean(jnp.array(y_train_raw, dtype=jnp.float64))
    target_std = jnp.std(jnp.array(y_train_raw, dtype=jnp.float64))
    target_std = jnp.where(target_std == 0, 1.0, target_std)

    # Apply normalization in float64 using JAX
    X_train_n = (jnp.array(X_train_raw, dtype=jnp.float64) - feat_mean) / feat_std
    y_train_n = (jnp.array(y_train_raw, dtype=jnp.float64) - target_mean) / target_std
    X_val_n = (jnp.array(X_val_raw, dtype=jnp.float64) - feat_mean) / feat_std
    y_val_n = (jnp.array(y_val_raw, dtype=jnp.float64) - target_mean) / target_std

    scaler = {
        "feature_cols": FEATURE_COLS,
        "feat_mean": np.array(feat_mean),
        "feat_std": np.array(feat_std),
        "target_mean": float(target_mean),
        "target_std": float(target_std),
        "normalized": True
    }

    # Cast to float32 ONLY for model input
    return np.array(X_train_n, dtype=np.float32), np.array(y_train_n, dtype=np.float32), \
           np.array(X_val_n, dtype=np.float32), np.array(y_val_n, dtype=np.float32), \
           y_val_raw, scaler

print("Preparing data (Normalized float64 SCALE)...")
X_train, y_train, X_val, y_val, y_val_raw, scaler = prepare_data(CONFIG)
print(f"Train size: {X_train.shape}, Val size: {X_val.shape}")

with open(CONFIG["scaler_path"], "wb") as f:
    pickle.dump(scaler, f)


Preparing data (Normalized float64 SCALE)...
Loading 773 wells...
Train size: (4054898, 6), Val size: (1037357, 6)


### 5. Build & Train Dense Model
Construct the MLP architecture and execute training with callbacks.

In [5]:
# Cell 5: Build & train Dense model

def build_dense_model(input_shape, cfg):
    """Builds a simple Multi-Layer Perceptron (MLP)."""
    inp = keras.Input(shape=input_shape)
    x = inp
    for units in cfg["hidden_layers"]:
        x = layers.Dense(units, activation="relu", kernel_regularizer=regularizers.L2(cfg["kr_rate"]))(x)
        if cfg["dropout"] > 0:
            x = layers.Dropout(cfg["dropout"])(x)
    
    out = layers.Dense(1, activation="linear")(x)
    m = keras.Model(inp, out, name="Dense_TVT")
    
    m.compile(
        optimizer=keras.optimizers.Adam(cfg["lr"], global_clipnorm=cfg["gcn"]),
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return m

model = build_dense_model((len(FEATURE_COLS),), CONFIG)
model.summary()

cbs = [
    callbacks.ModelCheckpoint(CONFIG["model_path"], monitor="val_rmse", save_best_only=True, mode="min"),
    callbacks.ReduceLROnPlateau(monitor="val_rmse", factor=0.5, patience=3, min_lr=1e-6),
    # callbacks.EarlyStopping(monitor="val_rmse", patience=7, restore_best_weights=True)
]

print(f"\nStarting training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=cbs
)


Model: "Dense_TVT"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,097 (78.50 KB)

 Trainable params: 20,097 (78.50 KB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/50
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 92s 3ms/step - loss: 0.0455 - rmse: 0.2088 - val_loss: 0.0460 - val_rmse: 0.2111 - learning_rate: 0.0010
Epoch 2/50
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 46s 1ms/step - loss: 0.0396 - rmse: 0.1953 - val_loss: 0.0369 - val_rmse: 0.1881 - learning_rate: 0.0010
Epoch 3/50
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 50s 2ms/step - loss: 0.0393 - rmse: 0.1944 - val_loss: 0.0456 - val_rmse: 0.2098 - learning_rate: 0.0010
Epoch 4/50
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 49s 2ms/step - loss: 0.0392 - rmse: 0.1940 - val_loss: 0.0334 - val_rmse: 0.1785 - learning_rate: 0.0010
Epoch 5/50
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 46s 1ms/step - loss: 0.0392 - rmse: 0.1939 - val_loss: 0.0484 - val_rmse: 0.2164 - learning_rate: 0.0010
Epoch 6/50
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 46s 1ms/step - loss: 0.0391 - rmse: 0.1938 - val_loss: 0.0441 - val_rmse: 0.2062 - learning_rate: 0.0010
Epoch 7/50
31679/31679 ━━━━━━━━━━━━━━━━━━━━ 46s 1ms/step - loss: 0.0391 - rmse: 0.19

### 6. Evaluate Model
Reload the best weights and calculate final validation RMSE on raw scale.

In [6]:
# Cell 6: Evaluate (RAW SCALE)

best_model = keras.saving.load_model(CONFIG["model_path"])
yp_n = best_model.predict(X_val, batch_size=1024).ravel()

# Denormalize using JAX
yp = yp_n * scaler["target_std"] + scaler["target_mean"]

err = yp - y_val_raw
rmse = float(np.sqrt(np.mean(err ** 2)))
print(f"\nFinal Val RMSE (Dense model, RAW SCALE): {rmse:.4f}")


1014/1014 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step

Final Val RMSE (Dense model, RAW SCALE): 113.7148
